# Тестовое задание на сбор и анализ данных для Сбера
**Ознакомиться с текстом тестового задания можно в файле `Тестовое задание.txt`.**
В этом ноутбуке будет представлен путь по очистке, обогащению и обезличиванию данных с использованием библиотеки pandas на python. 

Также, более подробно ознакомиться с описанием проделанной работы можно в файле `README.md`.

В самом начале требуется импорт нужных библиотек, настройка окружения, чтение файла с данными и краткое ознакомление с таблицей.

## Полный список юр.лиц
Здесь будет полный список ДЗО относящихся к группе компании Сбер.

In [ ]:
import pandas as pd

list_faces = pd.read_excel('list_of_entities.xlsx')

In [11]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.precision', 5)

tenders = pd.read_excel('sber_audit.xlsx')

tenders = tenders.drop_duplicates(subset=['Код процедуры'], keep='last')
tenders.notnull().count()
tenders.info()

tenders.head()

<class 'pandas.DataFrame'>
Index: 1364 entries, 0 to 1858
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Компания-заказчик              1364 non-null   str           
 1   Способ закупки                 1364 non-null   str           
 2   Код процедуры                  1364 non-null   object        
 3   Наименование предмета закупки  1364 non-null   str           
 4   Начальная сумма                1364 non-null   float64       
 5   Валюта                         1364 non-null   str           
 6   Статус закупки                 1364 non-null   str           
 7   Размещение (ЭТП)               1364 non-null   str           
 8   Дата публикации                1364 non-null   datetime64[us]
 9   Окончание подачи заявок        1364 non-null   datetime64[us]
 10  Дата подведения итогов         1364 non-null   datetime64[us]
 11  Тип закупки                    13

,Компания-заказчик,Способ закупки,Код процедуры,Наименование предмета закупки,Начальная сумма,Валюта,Статус закупки,Размещение (ЭТП),Дата публикации,Окончание подачи заявок,Дата подведения итогов,Тип закупки
0,АО «Сбербанк Лизинг»,Запрос предложений в электронной форме,79442871,Поставка бутилированной питьевой воды г. Волго...,0.0,RUB,Подача заявок,SberB2B,2025-12-19 07:56:00,2025-12-25 12:00:00,2025-12-25,Питание и вода
100,АО «Сбербанк Лизинг»,Публичное предложение (с ЭП),SBR065-2507310002.1,Публичное предложение на право заключения дого...,5492000.0,RUB,Не состоялся(-ась),Реализация имущества,2025-07-31 09:25:00,2025-10-09 09:59:00,2025-10-10,ПО и лицензии
200,АО «Сбербанк Лизинг»,Публичное предложение (с ЭП),SBR065-2505270005.1,Публичное предложение на право заключения дого...,1254000.0,RUB,Не состоялся(-ась),Реализация имущества,2025-05-27 09:53:00,2025-08-05 09:59:00,2025-08-06,ПО и лицензии
300,АО «Сбербанк Лизинг»,Запрос предложений в электронной форме,100315525,"Поставка продовольственных товаров (чай, кофе,...",0.0,RUB,Отменено,SberB2B,2025-03-20 17:17:00,2025-03-26 12:00:00,2025-03-26,Питание и вода
400,АО «Сбербанк Лизинг»,Запрос предложений в электронной форме,1161300,Оказание услуг по предоставлению ледовой площа...,0.0,RUB,Подача заявок,SberB2B,2025-02-11 13:29:00,2025-02-14 10:00:00,2025-02-14,ПО и лицензии


## Этап 1. Работа с сырыми данными
Первичный анализ показал, что с данными все хорошо: нет постуых ячееr и удалено 495 дублирующих записей. Итоговый размер таблицы 1364 записи.
В 1-ом этапе необходимо обогатить и обезличить данные.

В качестве обогащения будет добавлен номер ИНН. Вдобавок изменены цены, все приведено в единый формат.
Обезличивание данный не потребовалось, так как данные были взяты из источников, которые не публикуют информацию о личности/компании предоставляющей услуги.

In [ ]:
inn_mapping = {
    'АО «Сбербанк Лизинг»': '7707009586',
    'АО «СберТех»': '7736633354',
    'ООО «СалютДевайсы»': '7730253720',
    'ООО «ЮМани»': '7712015297',
    'ООО «Облачные технологии»': '7736324350',
    'ООО «Звук»': '7708323212',
    'АО «СберСервис»': '7730648052',
    'ООО «С-Маркетинг»': '7736319623',
    'АО «ОКБ»': '7710561081',
    'АО «Деловая среда»': '7704795744'
}
tenders['ИНН_Заказчика'] = tenders['Компания-заказчик'].map(inn_mapping)

tenders['Начальная сумма'] = (
    tenders['Начальная сумма']
    .astype(str)
    .str.replace('\xa0', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

In [ ]:
tenders.to_excel('all_sber_clean.xlsx', index=False)